[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_bloodchemistry.ipynb) [![Open In nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_bloodchemistry.ipynb)

# Blood chemistry

We just need two packages for this tutorial.

In [1]:
import pandas as pd
import pyaging as pya

## Download and load example data

30 real NHANES IV subjects, from the public `NHANES4` table of the R package [BioAge](https://github.com/dayoonkwon/BioAge).

In [2]:
pya.data.download_example_data('blood_chemistry_example')

⏺ example data already at pyaging_data/blood_chemistry_example.pkl

'pyaging_data/blood_chemistry_example.pkl'

In [3]:
df = pd.read_pickle('pyaging_data/blood_chemistry_example.pkl')

In [4]:
df.head()

,albumin,creatinine,glucose,c_reactive_protein,lymphocyte_percent,mean_cell_volume,red_cell_distribution_width,alkaline_phosphatase,white_blood_cell_count,total_cholesterol,blood_urea_nitrogen,hemoglobin_a1c,systolic_blood_pressure,forced_expiratory_volume,age,female
NHANES_2007_41475,36.0,45.968884,5.8830,1.98,20.0,92.1,13.4,113.0,8.5,4.62894,4.641,5.7,123.333333,2.025,62.0,1.0
NHANES_2007_41479,42.0,63.649224,5.4945,0.02,46.4,87.6,12.1,78.0,5.1,4.86168,4.284,5.7,108.666667,2.957,52.0,0.0
NHANES_2007_41482,45.0,61.881190,7.9365,0.40,22.3,89.4,12.4,85.0,9.4,4.08588,3.927,7.0,116.000000,2.920,64.0,0.0
NHANES_2007_41483,35.0,120.226312,5.9940,1.49,24.0,85.3,14.7,55.0,9.8,3.67212,6.069,6.5,109.333333,2.355,66.0,0.0
NHANES_2007_41486,43.0,54.809054,6.2160,0.39,36.3,90.7,11.7,101.0,5.6,5.01684,3.213,6.1,122.666667,2.364,61.0,1.0


`pya.utils.get_feature_ranges` shows the unit and plausible range for each feature.

In [ ]:
pya.utils.get_feature_ranges('phenoage')

,feature,unit,low,high
0,albumin,g/L,10.00,70.0
1,creatinine,umol/L,10.00,3000.0
2,glucose,mmol/L,1.00,60.0
3,c_reactive_protein,mg/dL,0.01,50.0
4,lymphocyte_percent,%,0.00,100.0
5,mean_cell_volume,fL,40.00,150.0
6,red_cell_distribution_width,%,8.00,40.0
7,alkaline_phosphatase,U/L,5.00,5000.0
8,white_blood_cell_count,10^3 cells/uL,0.05,500.0
9,age,years,0.00,122.5


## Convert data to AnnData object

AnnData objects are highly flexible and are thus our preferred method of organizing data for age prediction.

In [6]:
adata = pya.preprocess.df_to_adata(df)

Note that the original DataFrame is stored in `X_original` under layers. This is what the `adata` object looks like:

In [7]:
adata

AnnData object with n_obs × n_vars = 30 × 16
    var: 'percent_na'
    layers: 'X_original', None (.X)

## Predict age

LinAge2 is included deliberately even though this panel does not cover it, so you can see what an incomplete input looks like.

In [ ]:
pya.pred.predict_age(adata, ['phenoage', 'kdmage', 'homeostaticdysregulation'])

In [9]:
adata.obs.head()

,phenoage,kdmage,homeostaticdysregulation,linage2
NHANES_2007_41475,72.931468,54.722088,4.302934,60.837872
NHANES_2007_41479,45.493712,41.957925,3.207144,46.463145
NHANES_2007_41482,70.980976,48.563729,4.277576,56.877960
NHANES_2007_41483,86.424060,72.901056,4.843122,70.061409
NHANES_2007_41486,58.511456,47.659222,3.486106,54.568415


## Running LinAge2

LinAge2 needs 57 measurements **plus 26 questionnaire items**. Here are the two subjects the paper publishes, which carry the full panel.

In [ ]:
# The paper's two published example subjects, renamed to pyaging's feature names.
subjects = pd.DataFrame(
    {
        'told_high_blood_pressure': [1, 2],
        'told_diabetes': [1, 2],
        'general_health_condition': [3, 2],
        'health_compared_to_one_year_ago': [1, 3],
        'healthcare_visits_past_year': [3, 2],
        'hospital_overnight_past_year': [2, 2],
        'told_weak_or_failing_kidneys': [2, 2],
        'told_asthma': [2, 2],
        'treated_for_anemia_past_3_months': [2, 2],
        'told_arthritis': [2, 1],
        'told_congestive_heart_failure': [1, 2],
        'told_coronary_heart_disease': [2, 2],
        'told_angina': [1, 2],
        'told_heart_attack': [1, 2],
        'told_stroke': [2, 2],
        'told_emphysema': [2, 2],
        'told_thyroid_disease': [2, 2],
        'told_overweight': [1, 2],
        'told_chronic_bronchitis': [2, 2],
        'told_liver_condition': [2, 2],
        'told_cancer': [2, 2],
        'fractured_hip': [2, 2],
        'fractured_wrist': [2, 1],
        'fractured_spine': [2, 2],
        'told_osteoporosis': [2, 2],
        'confusion_or_memory_problems': [2, 2],
        'pulse': [68, 50],
        'systolic_blood_pressure': [111, 154],
        'diastolic_blood_pressure': [50, 74],
        'body_mass_index': [31.31, 23.13],
        'urine_albumin': [96.9, 8.9],
        'urine_creatinine': [18210, 13702],
        'iron': [19.87, 28.64],
        'total_iron_binding_capacity': [54.95, 61.04],
        'transferrin_saturation': [36.2, 46.9],
        'ferritin': [81, 292],
        'folate': [40.5, 60.2],
        'vitamin_b12': [418.45, 686.34],
        'cotinine': [182.83, 0.035],
        'total_cholesterol': [3.83, 5.72],
        'hdl_cholesterol': [0.95, 1.83],
        'white_blood_cell_count': [6.6, 6.8],
        'lymphocyte_percent': [26.5, 17.2],
        'monocyte_percent': [8.5, 8.1],
        'neutrophil_percent': [60.1, 72.2],
        'eosinophil_percent': [4.3, 2.2],
        'basophil_percent': [0.6, 0.4],
        'lymphocyte_count': [1.7, 1.2],
        'monocyte_count': [0.6, 0.6],
        'neutrophil_count': [4, 4.9],
        'eosinophil_count': [0.3, 0.1],
        'basophil_count': [0, 0],
        'red_blood_cell_count': [4.99, 4.81],
        'hemoglobin': [15.1, 15.4],
        'hematocrit': [46.6, 46.2],
        'mean_cell_volume': [93.2, 96.1],
        'mean_cell_hemoglobin': [30.3, 32],
        'mean_cell_hemoglobin_concentration': [32.5, 33.3],
        'red_cell_distribution_width': [13.2, 12.6],
        'platelet_count': [293, 264],
        'mean_platelet_volume': [8.1, 10],
        'c_reactive_protein': [0.95, 0.08],
        'hemoglobin_a1c': [8, 5.3],
        'nt_probnp': [1029, 72.14],
        'albumin': [43, 46],
        'alanine_aminotransferase': [15, 32],
        'aspartate_aminotransferase': [16, 26],
        'alkaline_phosphatase': [148, 107],
        'blood_urea_nitrogen': [3.9, 10],
        'calcium': [2.525, 2.225],
        'bicarbonate': [25, 28],
        'glucose': [3.941, 5.273],
        'lactate_dehydrogenase': [177, 196],
        'phosphorus': [1.518, 1.227],
        'total_bilirubin': [13.7, 10.3],
        'total_protein': [79, 72],
        'triglycerides': [1.027, 0.96],
        'uric_acid': [303.3, 291.5],
        'creatinine': [79.6, 70.7],
        'sodium': [142.4, 139.4],
        'potassium': [4.35, 4.08],
        'chloride': [101.6, 100.8],
        'globulin': [36, 26],
        'age': [72, 72.3333],
        'female': [0, 0],
    },
    index=['NHANES_8881', 'NHANES_9106'],
)

In [11]:
linage2_adata = pya.preprocess.df_to_adata(subjects, verbose=False)
pya.pred.predict_age(linage2_adata, 'LinAge2')
linage2_adata.obs

,linage2
sample,
NHANES_8881,88.694487
NHANES_9106,64.357899


## Checking units

As an example, albumin is reported in g/L by some labs and g/dL by others. `pyaging` expects g/L.

In [12]:
in_g_per_dl = df.copy()
in_g_per_dl['albumin'] = in_g_per_dl['albumin'] / 10  # g/L -> g/dL, the wrong unit for pyaging

wrong_units = pya.preprocess.df_to_adata(in_g_per_dl, verbose=False)
pya.pred.predict_age(wrong_units, 'PhenoAge')

The check is warn-only but it names the feature, the expected range, and what it saw.

In [13]:
comparison = pd.DataFrame({'albumin in g/L (correct)': adata.obs['phenoage'], 'albumin in g/dL (wrong)': wrong_units.obs['phenoage']})
comparison['error in years'] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
comparison.head()

,albumin in g/L (correct),albumin in g/dL (wrong),error in years
NHANES_2007_41475,72.931468,85.005332,12.073864
NHANES_2007_41479,45.493712,59.579887,14.086175
NHANES_2007_41482,70.980976,86.073306,15.092330
NHANES_2007_41483,86.424060,98.162539,11.738479
NHANES_2007_41486,58.511456,72.933016,14.421560


## Get citation

The doi, citation, and some metadata are automatically added to the AnnData object under `adata.uns[CLOCKNAME_metadata]`.

In [18]:
adata.uns['kdmage_metadata']

{'clock_name': 'kdmage',
 'data_type': 'clinical biomarkers',
 'species': 'Homo sapiens',
 'year': 2021,
 'approved_by_author': '⌛',
 'citation': 'Kwon, Dayoon, and Daniel W. Belsky. "A toolkit for quantification of biological age from blood chemistry and organ function test data: BioAge." GeroScience 43.6 (2021): 2795-2808.',
 'doi': 'https://doi.org/10.1007/s11357-021-00480-5',
 'notes': "Klemera-Doubal biological age, trained sex-specifically on NHANES III adults aged 30-75 who were not pregnant, using the BioAge package defaults. Biomarker parameters were fit on SI-unit variants so they are natively in pyaging's unit convention, and C-reactive protein is supplied raw in mg/dL and log1p-transformed inside the clock. Sex is coded female = 1 and male = 0; a dataset with no female column scores every sample with the male parameters.",
 'research_only': None,
 'version': None,
 'tissue': ['blood'],
 'predicts': ['biological age'],
 'training_target': ['chronological age'],
 'unit': ['ye